In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Caricamento e unione dati (android)
# Caricamento dei file "Median" (migliori)
df1 = pd.read_csv('android_session_1_1_median.csv')
df2 = pd.read_csv('android_session_2_2_median.csv')

# Unione in dataset per la Cross-Validation
df_full = pd.concat([df1, df2], ignore_index=True)

# Gestione valori mancanti (se ce ne sono ancora)
df_full = df_full.fillna(110)

print(f"Dataset Completo: {df_full.shape[0]} campioni")

# Separazione Feature (X) e Target (y)
# Esclusione colonne non-beacon (room, artwork, timestamp, ecc.)
# Si assume che le colonne beacon abbiano il trattino "-" (es. 12-1)
beacon_cols = [c for c in df_full.columns if '-' in c and c not in ['room', 'artwork']]
X = df_full[beacon_cols]
y_room = df_full['room']      # Target Stanza
y_artwork = df_full['artwork'] # Target Opera

# Configurazione modelli
# Usiamo i parametri migliori che avevo trovato
# k-NN Baseline
knn = KNeighborsClassifier(n_neighbors=57, metric='cityblock')

# Random Forest
# Uso i parametri ottimi
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Cross-validation (10-fold)
print("\nAvvio Cross-Validation (10-Fold)...")

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Calcolo le accuratezze (Score) per ogni fold
# Test sulla "Artwork Accuracy" che è la più difficile
scores_knn = cross_val_score(knn, X, y_artwork, cv=cv, scoring='accuracy', n_jobs=-1)
scores_rf = cross_val_score(rf, X, y_artwork, cv=cv, scoring='accuracy', n_jobs=-1)

# Analisi statistica (t-test)
print("\n=== RISULTATI CROSS-VALIDATION ===")
print(f"k-NN Accuracy:        {scores_knn.mean():.4f} (+/- {scores_knn.std():.4f})")
print(f"Random Forest Accuracy: {scores_rf.mean():.4f} (+/- {scores_rf.std():.4f})")

# T-Test Accoppiato (Paired T-Test)
# Ipotesi nulla (H0): I due algoritmi hanno la stessa performance media.
# Se p-value < 0.05, si rifiuta H0 -> La differenza è statisticamente significativa
t_stat, p_val = stats.ttest_rel(scores_knn, scores_rf)

print("\n=== ANALISI DI SIGNIFICATIVITÀ (T-TEST) ===")
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value:     {p_val:.4e}") # Notazione scientifica

if p_val < 0.05:
    print("RISULTATO: La differenza è STATISTICAMENTE SIGNIFICATIVA.")
    print("Il miglioramento NON è dovuto al caso.")
else:
    print("RISULTATO: La differenza NON è statisticamente significativa.")

plt.figure(figsize=(8, 6))
data_to_plot = pd.DataFrame({
    'k-NN': scores_knn,
    'Random Forest': scores_rf
})

sns.boxplot(data=data_to_plot, palette="Set2")
sns.swarmplot(data=data_to_plot, color=".25")

plt.title(f'Confronto Statistico (10-Fold CV)\np-value: {p_val:.2e}')
plt.ylabel('Artwork Accuracy')
plt.grid(True, alpha=0.3)
plt.savefig('Analisi_Statistica_CV.png', dpi=300)
plt.show()

results_df = pd.DataFrame({'Fold': range(1,11), 'kNN_Acc': scores_knn, 'RF_Acc': scores_rf})
results_df.to_csv('Risultati_Statistici_Detailed.csv', index=False)